# EVCharge Intelligence — Indian synthetic preprocessing

Validate the generated Indian schema through the existing module. Asia/Kolkata is explicitly declared by the CSV. Generator code, seed, distributions/loss assumptions and reuse licence were not supplied; the legacy raw CSV remains preserved.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

current = Path.cwd().resolve()
ROOT = current if (current / "src/data_preparation.py").is_file() else current.parent
if not (ROOT / "src/data_preparation.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_preparation import run_pipeline

report = run_pipeline(ROOT / "data/raw/EV_Dataset_India.csv",
                      ROOT / "data/processed", ROOT / "outputs/data_quality")
assert report["reconciliation"]["balanced"]
assert report["source_integrity"]["unchanged"]
print(json.dumps(report["reconciliation"], indent=2))

{
  "input_rows": 200000,
  "exact_duplicates_removed": 0,
  "rejected_rows": 0,
  "accepted_rows": 200000,
  "balanced": true
}


## Outputs and derived-feature preview

Accepted records have usable identifiers and timestamps under the preparation rules. Exact duplicate copies are removed first. Every remaining conflicting session-ID record, missing ID, invalid timestamp or end-before-start record is quarantined in `rejected_sessions.csv`.

Flags describe additional issues without deleting usable demand rows. `flagged_records.csv` includes both accepted and rejected records with reasons and source row numbers. Measurement validity status and nullable energy plausibility columns are available in accepted data too. Blank plausibility flags mean not evaluable. Full records stay in ignored CSV exports; the preview below contains only three rows of derived features.

In [2]:
features = list(report["derived_features"])
preview = pd.read_csv(ROOT / "data/processed/charging_sessions.csv", usecols=features, nrows=3)
print(preview[features].to_string(index=False))
print("\nFlagged rows:", report["flagged_rows"])
print("Station IDs with inconsistent mappings:", report["station_ids_with_multiple_values"])

Charging_Date  Charging_Hour  Weekday_Index Weekday_Name  Is_Weekend Year_Month  Calculated_Session_Duration_Min Arrival_Date  Arrival_Hour  Calculated_Queue_Time_Min  Calculated_Active_Charging_Duration_Min  Post_Charging_Occupancy_Min Station_Port_Key
   2024-05-14             19              1      Tuesday       False    2024-05                             53.0   2024-05-14            19                        0.0                                     51.0                          2.0         ST0879/1
   2024-07-05              8              4       Friday       False    2024-07                            191.0   2024-07-05             8                        0.0                                     20.0                        171.0         ST0861/3
   2025-10-24             15              4       Friday       False    2025-10                             36.0   2025-10-24            15                        0.0                                     34.0                          2.0  

## Calendar and time features

All four timestamps are explicitly parsed day-first and localized to Asia/Kolkata. Original exports remain DD-MM-YYYY HH:MM plus Timezone. Derived start/arrival features and separate calculated queue/active/occupancy/post-charge minutes are recomputed deterministically.

Accepted session starts span 01-01-2024 01:53 to 30-12-2025 23:53 in Asia/Kolkata, across 730 inclusive dates. Complete collection including zero-session dates is an assumption; timestamps do not establish uptime or complete boundary-day observation.

In [3]:
print(json.dumps(report["coverage"], indent=2))

{
  "earliest_session_start": "01-01-2024 01:53",
  "latest_session_start": "30-12-2025 23:53",
  "timezone": "Asia/Kolkata",
  "first_date": "2024-01-01",
  "last_date": "2025-12-30",
  "inclusive_calendar_days": 730,
  "observed_session_dates": 730,
  "calendar_average_assumption": "Future averages assume complete collection across the inclusive range, including zero-session dates.",
  "limitation": "Observed timestamps do not verify collection uptime or complete first/last-day coverage. Boundary months may be partial; timezone is Asia/Kolkata for the supplied synthetic dataset."
}


## Measurements and port validation

Finite nonnegative energy/queue values are numerically valid. Energy capacity and nominal **active** charger energy are separate plausibility checks; SOC energy and durations are also validated. Half-open occupancy intervals must not overlap on a station-local port; every conflict participant is flagged. Structural timestamp/port/timezone errors are quarantined, not corrected.

Capacity and active-charger-energy concerns each flag 0 sessions; the union is 0. Sensitivity retains all 200,000 sessions (100%) and 3,014,172.81 kWh (100% of baseline). Curves coincide because nothing is excluded. This does not verify real measurements: this is synthetic consistency, not corrected/ground-truth data.

In [4]:
print(json.dumps(report["measurement_validity"], indent=2))
print(json.dumps(report["plausibility_concerns"], indent=2))
print("Raw SHA-256 unchanged:", report["source_integrity"]["unchanged"])
print("SHA-256:", report["source_integrity"]["sha256_after"])

{
  "Energy_Delivered_kWh": {
    "after_deduplication": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    },
    "accepted": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    }
  },
  "Queue_Time_Min": {
    "after_deduplication": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    },
    "accepted": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    }
  }
}
{
  "Energy_Exceeds_Battery_Capacity": {
    "flagged_after_deduplication": 0,
    "flagged_accepted": 0,
    "not_evaluable_after_deduplication": 0,
    "not_evaluable_accepted": 0
  },
  "Energy_Exceeds_Nominal_Charger_Energy": {
    "flagged_after_deduplication": 0,
    "flagged_accepted": 0,
    "not_evaluable_after_deduplication": 0,
    "not_evaluable_accepted": 0
  }
}
Raw SH

## Handoff limitations

The input is generated synthetic Indian charging data. Station mappings are internally consistent in this run, not independently verified locations. Generator code, seed, distribution/efficiency assumptions and reuse licence were not supplied. Keep arrivals, active charging and port occupancy separate. Counts alone do not prove congestion; port counts do not establish operational availability.

All 200,000 energy measurements are numerically valid: 3,014,172.81 kWh total, 15.07 kWh per valid-energy session. Monthly energy/day ranges from 3,327.97 kWh in 2024-09 to 4,929.07 kWh in 2025-04. Full energy belongs to the session-start period, not measured hourly grid load.

See docs/dataset_migration.md and docs/data_dictionary.md for field definitions and Power BI changes.